# 03 — Sequence Models: MLP, LSTM & Hybrid (PyTorch)
Train and compare three neural network architectures on temporal
windows of SOLEY PV data:

| Architecture | Description |
|---|---|
| **MLP**    | Per-timestep features from the last step of the window |
| **LSTM**   | Bi-directional LSTM over the full window |
| **Hybrid** | MLP branch (last step) + LSTM branch fused (recommended) |

The streaming memmap pipeline keeps peak RAM ≈ 1 parquet file (~60–100 MB)
regardless of dataset size.


## 1. Imports & setup

In [1]:
import sys, pathlib, shutil
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from library.utils      import setup_logging, get_device, get_num_workers
from library.config     import BatchConfig
from library.data       import prepare_file_registry, assign_splits
from library.models.trainer import run_pytorch_task

setup_logging()
%matplotlib inline
plt.rcParams["figure.dpi"] = 120


## 2. Configuration

In [2]:
DATA_DIR   = "data"
OUTPUT_DIR = "outputs/sequence_models"
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Training hyperparameters
WINDOW_SIZE = 2016   # timesteps per sample (7 days × 24 h × 12 steps/h)
             # A 7-day context is realistic for fault monitoring:
             # gradual degradation (soiling, aging, EL darkening) manifests
             # over several days; an LSTM can learn those multi-day trends.
STRIDE      = 288    # step between windows (1 day = 288 × 5-min steps)
             # 1-day stride avoids excessive window overlap while still
             # producing ~350 training windows per 1-year run.
BATCH_SIZE  = 512    # reduced from 1024 because each sample is ~4× larger
EPOCHS      = 30
PATIENCE    = 7
MAX_RUNS    = None     # must match 02_random_forest.ipynb for fair comparison

# Model variants to train: "mlp", "lstm", "hybrid", or all three
MODEL_MODES = ["mlp", "lstm", "hybrid"]


In [3]:
cfg    = BatchConfig(DATA_DIR)
device = get_device()
n_workers = get_num_workers()
print(f"Device:  {device}")
print(f"Workers: {n_workers}")
print(cfg)


  Config: array=5.34 kWp, 6 location(s)
  God-mode columns found in data and excluded from all feature sets: ['detailed_balance_efficiency_pct', 'ff', 'jmpp_a_m2', 'jsc_a_m2', 'vmpp_v', 'voc_v']
  Columns: 11 SCADA, 8 device physics, 6 stress, 29 constant (excluded), 6 god-mode (excluded), 25 total features
Device: Apple MPS
Device:  mps
Workers: 4
BatchConfig(array_kwp=5.34, locations=6, features=25)


## 3. Build file registry & assign splits

In [4]:
# No data is loaded here — just file paths and fault-type metadata
registry = prepare_file_registry(DATA_DIR, cfg, max_runs=MAX_RUNS)
assign_splits(registry)

fault_types = sorted({e['fault_type'] for e in registry})
print(f"\nFault types in registry: {fault_types}")


Registry: 310 files, 12 fault types
  train: 200 files
  val: 55 files
  test: 55 files

Fault types in registry: ['bypass_diode_short', 'cell_crack', 'combo', 'connector_burnout', 'curtailment', 'mppt_failure', 'none', 'pid', 'sensor_drift', 'solder_fatigue', 'string_failure', 'sudden_soiling']


## 4. Feature list

In [5]:
# Use the same feature set as 02_random_forest.ipynb — identical call to
# cfg.feature_set("full") ensures both model families see the same inputs.
feature_list = cfg.feature_set("full")
print(f"Total features: {len(feature_list)}")


Total features: 33


## 5. Task A — Fault Detection (binary)

In [ ]:
detection_reports = run_pytorch_task(
    task_name   = "Fault Detection",
    target_col  = "fault_active",
    registry    = registry,
    feature_cols= feature_list,
    model_modes = MODEL_MODES,
    window_size = WINDOW_SIZE,
    stride      = STRIDE,
    batch_size  = BATCH_SIZE,
    epochs      = EPOCHS,
    device      = device,
    output_dir  = OUTPUT_DIR,
    patience    = PATIENCE,
    cfg         = cfg,
    num_workers = n_workers,
)



  Fault Detection
  Classes: 2  Files: train=200, val=55, test=55
  Fitting scaler …
After daytime filter: 306,529 rows
After daytime filter: 299,039 rows
After daytime filter: 306,851 rows
After daytime filter: 306,824 rows
After daytime filter: 307,917 rows
After daytime filter: 307,688 rows
After daytime filter: 307,372 rows
After daytime filter: 307,514 rows
After daytime filter: 306,645 rows
After daytime filter: 306,779 rows
After daytime filter: 302,896 rows
After daytime filter: 302,712 rows
After daytime filter: 303,046 rows
After daytime filter: 299,627 rows
After daytime filter: 298,802 rows
After daytime filter: 306,782 rows
After daytime filter: 306,634 rows
After daytime filter: 306,414 rows
After daytime filter: 306,802 rows
After daytime filter: 306,885 rows
  Scaler fit on 20 files, 33 features
  Building train arrays (200 files) …
After daytime filter: 306,529 rows
After daytime filter: 306,678 rows
After daytime filter: 306,917 rows
After daytime filter: 307,835 row

In [ ]:
# Training curves
from IPython.display import Image, display as ipy_display
for mode in MODEL_MODES:
    p = f"{OUTPUT_DIR}/curves_fault_detection_{mode}.png"
    if pathlib.Path(p).exists():
        print(f"\n{mode.upper()} training curves:")
        ipy_display(Image(p))


## 6. Task B — Fault Classification (multi-class)

In [ ]:
classification_reports = run_pytorch_task(
    task_name   = "Fault Classification",
    target_col  = "fault_type",
    registry    = registry,
    feature_cols= feature_list,
    model_modes = MODEL_MODES,
    window_size = WINDOW_SIZE,
    stride      = STRIDE,
    batch_size  = BATCH_SIZE,
    epochs      = EPOCHS,
    device      = device,
    output_dir  = OUTPUT_DIR,
    patience    = PATIENCE,
    cfg         = cfg,
    num_workers = n_workers,
)


In [ ]:
# Per-fault F1 comparison
p = f"{OUTPUT_DIR}/per_fault_f1_fault_classification.png"
if pathlib.Path(p).exists():
    ipy_display(Image(p))


## 7. Confusion matrices

In [ ]:
for task_tag in ["fault_detection", "fault_classification"]:
    for mode in MODEL_MODES:
        p = f"{OUTPUT_DIR}/confusion_{task_tag}_{mode}.png"
        if pathlib.Path(p).exists():
            print(f"\n{task_tag} — {mode.upper()}")
            ipy_display(Image(p))


## 8. Saved artifacts

In [ ]:
import os
outputs = sorted(pathlib.Path(OUTPUT_DIR).iterdir())
for f in outputs:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name:<50s}  {size_mb:.2f} MB")
